In [ ]:
%%sql -r Uses
USE DATABASE STAR_DEV;
USE SCHEMA PROFILE;

# PATIENT MASTERING — CORE Methods

This notebook illustrates algorithmic logic that makes patient mastering work. Pipeline scaffolding, warehouses, RBAC, streams, tasks, audit plumbing, and DDL for infrastructure tables have not been considered.

What is here:

| Section Number | Title    | Description |
|--|--------------------------------|----------------------------------|  
|#1|  Standardization transforms    | improve raw values 'matchability'|
|#2|  Blocking key generation       | determine the candidate match space|
|#3|  Per-field scoring             | similarity computation|
|#4|  Composite scoring + decision  | scores become match/review/no-match|
|#5|  Survivorship                  | resolve conflicting records to one truth|
|#6|  Data quality rules            | detect and classify data issues| 
|#7|  Blocking quality metrics      | evaluate blocking key choices|
|#8|  Block skew diagnostics        | evaluate blocking key choice on compute|
|#9|  Singleton detection           | handle records not like th others|
|#10| Ground truth bootstrap        | find a set of highly confident 'true matches'|


# #1  STANDARDIZATION TRANSFORMS

**Input:**  Raw VARCHAR fields exactly as delivered by the source system

**Output:** Clean, comparable values + phonetic keys for blocking


In [ ]:
%%sql -r STANDARDIZE
CREATE OR REPLACE TEMPORARY TABLE STAN_NORM AS
WITH 
-- Identify Patients that don't appear in key fact tables
PATIENT_SRC AS ( 
    SELECT DISTINCT
        'APPOINTMENT' AS SRC,
        1 AS MARKER_FLAG,
        DIM_PATIENT_KEY,
    FROM
        STAR.DIM_MODEL.FACT_APPOINTMENT
    UNION
    SELECT
        'LEDGER',
        1,
        DIM_PATIENT_KEY
    FROM
        STAR.DIM_MODEL.FACT_BILLING_LEDGER
    UNION
    SELECT
        'ENCOUNTER',
        1,
        DIM_PATIENT_KEY
    FROM
        STAR.DIM_MODEL.FACT_ENCOUNTER_PROCEDURE
   UNION
    SELECT
        'RECALL',
        1,
        DIM_PATIENT_KEY
    FROM
        STAR.DIM_MODEL.FACT_RECALL 
   UNION
   SELECT
       'DIM_PATIENT',
       1,
       DIM_PATIENT_KEY
   FROM 
       STAR.DIM_MODEL.DIM_PATIENT 
),
TO_MATRIX AS(
    SELECT 
        *
    FROM 
        PATIENT_SRC
        PIVOT(COUNT(MARKER_FLAG) FOR SRC IN (ANY ORDER BY SRC))
    ORDER BY 
        DIM_PATIENT_KEY
),
-- Remove punctuation, control chars, zero-width spaces
CLEAN AS (
    SELECT
        DIM_PATIENT_KEY,
        FIRST_NAME,
        -- Identity: First Name
        --First Name Standardization
        --      Collapse multiple spaces
        --      Strip digits
        --      Strip non-sensical punctuation; keep hyphen, apostrophe, period
        --      Collapse repeated hyphens, apostrophes, periods
        --      Strip leading/trailing hyphens, apostrophes, periods
        --      Strip trailing period (initial like "J.")
        --      Proper-case each hyphen-separated segment
        normalize_first_name(FIRST_NAME) AS FIRST_NAME_cln,
        LAST_NAME,
        -- Identity: Last Name
        -- Last Name Standardization
        --      Collapse multiple spaces
        --      Strip digits
        --      Strip non-sensical punctuation; keep hyphen, apostrophe, period
        --      Collapse repeated hyphens, apostrophes, periods
        --      Strip leading/trailing hyphens, apostrophes, periods
        --      Split on hyphen, apply Mc / O' / standard proper-case per segment
        --      Normalize suffixes via CASE (avoids backreference regex)
        --      Proper-case each hyphen-separated segment
        normalize_last_name(LAST_NAME) AS LAST_NAME_cln,
        PATIENT_DOB,
        PATIENT_GENDER,
        PATIENT_ADDRESS1,
        -- Contact: Address1
        -- Uppercases 
        -- Trims
        -- Collapse whitespace
        -- Remove punctuation
        -- Exception: hyphens (for hyphenated house numbers like 12-14)
        cleanse_street_address(PATIENT_ADDRESS1) AS PATIENT_ADDRESS1_cln,
        PATIENT_CITY,
        TRIM(REGEXP_REPLACE(PATIENT_CITY,'[^A-Za-z\\s]',' ')) AS PATIENT_CITY_cln,
        PATIENT_STATE,
        TRIM(REGEXP_REPLACE(PATIENT_STATE,'[^A-Za-z\\s]',' ')) AS PATIENT_STATE_cln,
        SUBSTR(REGEXP_REPLACE(PATIENT_ZIPCODE, '[^0-9]', ''), 1, 5) AS PATIENT_ZIPCODE,
        MRN,
        IDENTIFIER,
        ETL_CREATED_DATE
    FROM STAR.DIM_MODEL.DIM_PATIENT
),
STANDARDIZED AS (
    SELECT
        'UNMATCHED' AS MATCH_STATUS,
        DIM_PATIENT_KEY,
        FIRST_NAME,
        SUBSTR(FIRST_NAME,1,1) AS FN_INIT,
        LAST_NAME,
        SUBSTR(LAST_NAME,1,1) AS LN_INIT,
        SUBSTR(LAST_NAME,1,4) AS LN_SUB4,
        PATIENT_DOB,
        YEAR(PATIENT_DOB) AS YOB,
        PATIENT_GENDER,
        PATIENT_ADDRESS1,
        PATIENT_CITY,
        PATIENT_STATE,
        PATIENT_ZIPCODE,
        SUBSTR(PATIENT_ZIPCODE,1,3) AS ZIP3,
        MRN,
        IDENTIFIER,
        ETL_CREATED_DATE,
        FIRST_NAME_cln AS fn_std,
        LAST_NAME_cln AS ln_std,
        -- Phonetic blocking keys
        --  SOUNDEX maps names that sound alike to the same 4-character code.
        --  'SMITH', 'SMYTH', 'SMITHE' all produce S530.
        --  These columns are used in §2 to define blocking groups, not for
        --  scoring. They are intentionally lossy — that is the point.
        SOUNDEX(INITCAP(REGEXP_REPLACE(FIRST_NAME_cln, '\\s+', ' '))) AS FN_SDX,
        SOUNDEX(INITCAP(REGEXP_REPLACE(LAST_NAME_cln, '\\s+', ' '))) AS LN_SDX
        ,"'APPOINTMENT'","'LEDGER'","'ENCOUNTER'","'DIM_PATIENT'",
        
        -- Identity DQ  
        --  This is a primative proactive data-issue catcher - may go elsewhere.
        CASE 
          WHEN REGEXP_LIKE(FIRST_NAME, '(DUPLICATE|TEST|TESTING|DUMMY|SAMPLE|UNKNOWN|PATIENT|DEMO|ANONYMOUS|ANON|DO\\s?NOT\\s?USE|DELETE|FAKE|TEMP|PLACEHOLDER|NULL|N/A|NONE|XX+|ZZ+|QQ+|YY+)', 'i')
            OR REGEXP_LIKE(LAST_NAME, '(DUPLICATE|TEST|TESTING|DUMMY|SAMPLE|UNKNOWN|PATIENT|DEMO|ANONYMOUS|ANON|DO\\s?NOT\\s?USE|DELETE|FAKE|TEMP|PLACEHOLDER|NULL|N/A|NONE|XX+|ZZ+|QQ+|YY+)', 'i')
          THEN 1 
          ELSE 0 
          END AS FAUX_NAME,     
        CASE 
          WHEN REGEXP_LIKE(FIRST_NAME, '^\\s*$')
            OR REGEXP_LIKE(LAST_NAME, '^\\s*$')
          THEN 1 
          ELSE 0 
          END AS NAME_BLANK,
        CASE 
            WHEN PATIENT_DOB>=CURRENT_DATE()
            THEN 1
            ELSE 0
            END AS FUTURE_DOB,
        CASE
            WHEN PATIENT_DOB<'1901-01-31'::DATE
            THEN 1
            ELSE 0
            END AS FAUX_DATE,
        CASE 
          WHEN  REGEXP_LIKE(PATIENT_ZIPCODE,'\\d{5}')
          THEN 0
          ELSE 1
          END AS BAD_ZIP
    FROM CLEAN
    LEFT JOIN TO_MATRIX
    USING(DIM_PATIENT_KEY)
)
SELECT
    MATCH_STATUS,
    DIM_PATIENT_KEY
    --Blocking Analysis Columns
    ,ETL_CREATED_DATE,LN_STD,LN_INIT,LN_SUB4,FN_STD,FN_INIT,FN_SDX,LN_SDX,PATIENT_DOB,YOB,PATIENT_GENDER,PATIENT_ADDRESS1,PATIENT_CITY,PATIENT_STATE,PATIENT_ZIPCODE,ZIP3,
    MRN,IDENTIFIER,
    --Ground Truth Pair Designation Columns
    FIRST_NAME,LAST_NAME,
    -- DQ Support
    FAUX_NAME,NAME_BLANK,FUTURE_DOB,FAUX_DATE,BAD_ZIP
FROM 
    STANDARDIZED
--- Effectively Filter 'ghost patients'
WHERE "'APPOINTMENT'"=1 OR "'LEDGER'"=1 OR "'ENCOUNTER'"=1
;

In [ ]:
SELECT 'PAT KEY COUNT: ',COUNT(*) FROM STAN_NORM;

In [ ]:
%%sql -r dataframe_7
SELECT 
    FAUX_NAME,NAME_BLANK,FUTURE_DOB,FAUX_DATE,BAD_ZIP
    ,COUNT(*) AS FREQUENCY
FROM
    STAN_NORM
GROUP BY
   FAUX_NAME,NAME_BLANK,FUTURE_DOB,FAUX_DATE,BAD_ZIP
ORDER BY
    FREQUENCY DESC;

# #2  BLOCKING KEY GENERATION
***Goal:*** 

Generate only the candidate pairs worth scoring.
           Do NOT compare every record to every other record — that is O(n²) and will not complete on any real dataset.

***Strategy:*** 

Disjunctive blocking — a pair enters the candidate set if it matches on ANY of the blocking keys. More keys = higher recall
(Pairs Completeness) at a small efficiency cost (Reduction Ratio).

Passes are ordered: PRIMARY → FALLBACK → LAST_RESORT.
Records without any usable key end up in #9 (singletons).

Tuning these JOIN conditions is the highest-leverage decision in this entire system. See #7 for how to measure whether they are right.

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE TEMPORARY TABLE BLOCKED AS
WITH
--   Last Name same SOUNDEX(last_name)
--   --Catches: Smith / Smyth / Schmidt, transpositions, minor typos
--   --ZIP Code
--   Intial Analysis shows 51,283,579 with 0.99 Reduction Rate and 1.0 Pairs Complete
--   a.STD_RECORD_ID < b.STD_RECORD_ID 
PRIMARY_BLOCK AS (
    SELECT
        A.DIM_PATIENT_KEY     AS REC_A,
        B.DIM_PATIENT_KEY     AS REC_B,
        --   Last Name same SOUNDEX(last_name)
        --   --Catches: Smith / Smyth / Schmidt, transpositions, minor typos
        --   --ZIP Code
        --   Initial Analysis shows 51,283,579 with 0.99 Reduction Rate and 1.0 Pairs Complete
        'LNSDX_ZIP'         AS BLOCK_PASS
    FROM STAN_NORM AS A
    JOIN STAN_NORM AS B
        ON  A.DIM_PATIENT_KEY      < B.DIM_PATIENT_KEY          -- No self join and ensures each pair appears once.
        AND A.MATCH_STATUS       IN ('UNMATCHED', 'REVIEW')
        AND B.MATCH_STATUS       IN ('UNMATCHED', 'REVIEW')
        AND A.LN_SDX  IS NOT NULL
        AND B.LN_SDX  IS NOT NULL
        AND A.PATIENT_ZIPCODE    IS NOT NULL
        AND B.PATIENT_ZIPCODE    IS NOT NULL
        AND A.LN_SDX = B.LN_SDX
        AND A.PATIENT_ZIPCODE = B.PATIENT_ZIPCODE 
),
FALLBACK_BLOCK AS (
    SELECT
        A.DIM_PATIENT_KEY     AS REC_A,
        B.DIM_PATIENT_KEY     AS REC_B,
        --   Last Name same SOUNDEX(last_name)
        --   --Catches: Smith / Smyth / Schmidt, transpositions, minor typos
        --   --DOB
        --   Initial Analysis shows 2,758,083 with 0.99 Reduction Rate and 1.0 Pairs Complete
        'LNSDX_DOB'         AS BLOCK_PASS
    FROM STAN_NORM AS A
    JOIN STAN_NORM AS B
        ON  A.DIM_PATIENT_KEY      < B.DIM_PATIENT_KEY          
        AND A.MATCH_STATUS       IN ('UNMATCHED', 'REVIEW')
        AND B.MATCH_STATUS       IN ('UNMATCHED', 'REVIEW')
        -- at least one record has no usable last name key
        AND A.LN_SDX  IS NOT NULL
        AND B.LN_SDX  IS NOT NULL
        AND A.PATIENT_DOB      IS NOT NULL
        AND B.PATIENT_DOB      IS NOT NULL
        AND A.LN_SDX = B.LN_SDX
        AND A.PATIENT_DOB = B.PATIENT_DOB
),
LAST_RESORT_BLOCK AS (
    SELECT
        A.DIM_PATIENT_KEY     AS REC_A,
        B.DIM_PATIENT_KEY     AS REC_B,
        -- Initial Analysis shows 29,840,529 with 0.99 Reduction Rate and 1.0 Pairs Complete
        'ZIP3_DOB'  AS BLOCK_PASS
    FROM STAN_NORM AS A
    JOIN STAN_NORM AS B
        ON  A.DIM_PATIENT_KEY      < B.DIM_PATIENT_KEY
        AND A.MATCH_STATUS       IN ('UNMATCHED', 'REVIEW')
        AND B.MATCH_STATUS       IN ('UNMATCHED', 'REVIEW')
        AND (A.LN_SDX  IS NULL OR B.LN_SDX  IS NULL)
        AND A.PATIENT_ZIPCODE    IS NOT NULL
        AND B.PATIENT_ZIPCODE    IS NOT NULL
        AND A.PATIENT_DOB      IS NOT NULL
        AND B.PATIENT_DOB      IS NOT NULL
        AND A.PATIENT_DOB       = B.PATIENT_DOB  
        AND A.PATIENT_ZIPCODE            = B.PATIENT_ZIPCODE
),
COMBINE_PASSES AS (
    SELECT * 
    FROM PRIMARY_BLOCK
    UNION ALL
    SELECT * 
    FROM FALLBACK_BLOCK
    UNION ALL
    SELECT * 
    FROM LAST_RESORT_BLOCK
)
--Prioritize Block Pairs
-- Keep Block Source for tracking purposes
SELECT 
    REC_A, 
    REC_B, 
    BLOCK_PASS
    FROM (
        SELECT *, 
           ROW_NUMBER() OVER (
               PARTITION BY REC_A, REC_B 
               ORDER BY 
                   CASE BLOCK_PASS 
                       WHEN 'LNSDX_ZIP' THEN 1
                       WHEN 'LNSDX_DOB' THEN 2
                       ELSE 3
                   END
           ) AS rn
    FROM COMBINE_PASSES
)
WHERE rn = 1;

In [ ]:
SELECT DISTINCT REC_A FROM BLOCKED
UNION
SELECT DISTINCT REC_B FROM BLOCKED;


# #3  PER-FIELD SCORING

***Input:***  a candidate pair (REC_A, REC_B) from #2

***Output:*** a similarity score 0.0–1.0 per field

Each CASE expression is an independent scoring function.
They are combined in #4 using configurable weights.

     Notes on EDITDISTANCE():
       Snowflake's built-in function returns the minimum number of
       single-character edits (insertions, deletions, substitutions)
       needed to transform one string into another (Levenshtein distance).
       We normalize by the longer string's length to get a 0–1 similarity.
       This approximates Jaro-Winkler without a UDF:
         similarity = 1 - (edit_distance / max_length)
       For very short strings (length 1-2) this is unreliable; the
       NULLIF guard prevents division by zero.

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE FIELD_SCORES AS
WITH
SCORED AS (
    SELECT
        bp.REC_A,
        bp.REC_B,
        bp.BLOCK_PASS,
        
        -- SCORE_DOB 
        --  Exact match scores 1.0.
        --  Off-by-one-day scores 0.80 — catches the common day/month swap
        --  where the transposition produces a date one day off (e.g., the
        --  31st of one month parsed as the 1st of the next).
        --  Larger differences score 0.
        CASE
            WHEN a.PATIENT_DOB IS NULL OR b.PATIENT_DOB IS NULL THEN 0.0
            WHEN a.PATIENT_DOB = b.PATIENT_DOB                  THEN 1.0
            WHEN ABS(DATEDIFF('day', a.PATIENT_DOB,
                                     b.PATIENT_DOB)) <= 1         THEN 0.80
            ELSE 0.0
        END AS SCORE_DOB,

        -- SCORE_FNAME 
        --  Edit-distance similarity on the standardized first name.
        --  Handles: 'JOHN' vs 'JON', 'JOHNATHAN' vs 'JONATHAN'.
        --  Does NOT handle nickname → legal name ('BOB' vs 'ROBERT') —
        --  for that you need a nickname lookup table applied in §1.

        CASE
            WHEN a.fn_std IS NULL OR b.fn_std IS NULL THEN 0.0
            ELSE
                1.0 - (
                    EDITDISTANCE(a.fn_std, b.fn_std)::FLOAT
                    / NULLIF(GREATEST(LENGTH(a.fn_std),
                                      LENGTH(b.fn_std)), 0)
                )
        END AS SCORE_FNAME,

        -- SCORE_LNAME 
        --  Same pattern as SCORE_FNAME.
        --  Handles: 'SMITH' vs 'SMYTH', 'GONZALEZ' vs 'GONZALES'.
        --  SOUNDEX blocking in §2 already pre-filters for phonetic similarity,
        --  so most pairs reaching this score already share a Soundex code.
        --  This score refines within that phonetic bucket.

        CASE
            WHEN a.ln_std IS NULL OR b.ln_std IS NULL THEN 0.0
            ELSE
                1.0 - (
                    EDITDISTANCE(a.ln_std, b.ln_std)::FLOAT
                    / NULLIF(GREATEST(LENGTH(a.ln_std),
                                      LENGTH(b.ln_std)), 0)
                )
        END AS SCORE_LNAME,

        -- SCORE_GENDER 
        --  Binary. Low weight in §4 because gender can be recorded
        --  inconsistently across systems and updated over time.
        --  'U' (unknown) does not penalize — treated as missing, not mismatch.

        CASE
            WHEN a.PATIENT_GENDER IS NULL OR b.PATIENT_GENDER IS NULL              THEN 0.0
            WHEN a.PATIENT_GENDER = 'Unknown'    OR b.PATIENT_GENDER = 'Unknown'   THEN 0.0
            WHEN a.PATIENT_GENDER = b.PATIENT_GENDER                               THEN 1.0
            ELSE 0.0
        END AS SCORE_GENDER,

        -- SCORE_ZIP 
        --  Exact 5-digit ZIP match only.
        --  Partial ZIP matching (first 3 digits) is intentionally not used:
        --  it would create massive false-positive rates in dense metro areas.
        --  ZIP is a corroborating signal, not a primary one.

        CASE
            WHEN a.PATIENT_ZIPCODE IS NULL OR b.PATIENT_ZIPCODE IS NULL THEN 0.0
            WHEN a.PATIENT_ZIPCODE = b.PATIENT_ZIPCODE                  THEN 1.0
            ELSE 0.0
        END AS SCORE_ZIP,

        -- SCORE CITY
        CASE
            WHEN a.PATIENT_CITY IS NULL OR b.PATIENT_CITY IS NULL THEN 0.0
            WHEN a.PATIENT_CITY = b.PATIENT_CITY                  THEN 1.0
            ELSE 0.0
        END AS SCORE_CITY,
        
        -- SCORE STATE
        CASE
            WHEN a.PATIENT_STATE IS NULL OR b.PATIENT_STATE IS NULL THEN 0.0
            WHEN a.PATIENT_STATE = b.PATIENT_STATE                  THEN 1.0
            ELSE 0.0
        END AS SCORE_STATE,

    FROM BLOCKED AS bp
    JOIN STAN_NORM AS a ON a.DIM_PATIENT_KEY = bp.REC_A
    JOIN STAN_NORM AS b ON b.DIM_PATIENT_KEY = bp.REC_B 
)
SELECT * FROM SCORED;


In [ ]:
%%sql -r dataframe_6
SELECT
    SCORE_DOB, SCORE_FNAME, SCORE_LNAME, SCORE_GENDER, SCORE_ZIP, SCORE_STATE, SCORE_CITY
    ,COUNT(*) AS FREQUENCY
FROM 
    FIELD_SCORES
GROUP BY
    SCORE_DOB, SCORE_FNAME, SCORE_LNAME, SCORE_GENDER, SCORE_ZIP, SCORE_STATE, SCORE_CITY
ORDER BY
    FREQUENCY DESC;

## #4  COMPOSITE SCORING + THREE-TIER DECISION
***Input:***  Per-field scores from #3

***Output:*** Composite score 0.0–1.0 + decision label

Weights below reflect a standard healthcare MDM weighting scheme.

They MUST be externalized to a configuration table in production — do not hard-code them in application logic. Different source system combinations warrant different weight sets.

***Deterministic override:*** 
        If QQQQ OR PPP, the composite score is forced to 1.0 and the decision is AUTO_MATCH regardless of other field scores. Deterministic beats probabilistic. Always.

Three-tier decision:
       >= 0.85  → AUTO_MATCH     (no human needed)
       0.65–0.85 → REVIEW        (steward queue)
       < 0.65   → AUTO_NO_MATCH  (not the same patient)

These thresholds are calibrated against the ground truth set.
The values here are dynamic, not universal constants.

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE TEMPORARY TABLE COMP_DECISION AS
SELECT
    REC_A,
    REC_B,
    BLOCK_PASS,

    -- ── Per-field score breakdown ────────────────────────────────────────────
    --  Stored as a VARIANT so stewards can see why a pair scored as it did.
    OBJECT_CONSTRUCT(
        'dob',    SCORE_DOB,
        'fname',  SCORE_FNAME,
        'lname',  SCORE_LNAME,
        'gender', SCORE_GENDER,
        'zip',    SCORE_ZIP,
        'state',  SCORE_STATE,
        'city',   SCORE_CITY
    ) AS SCORE_BREAKDOWN,

    -- Weighted composite score 
    --  Weights sum to 1.0
    --
    --  Weight rationale:
    --    DOB    0.30  — high specificity, low corruption rate in EHR systems
    --    LNAME  0.20  — high specificity, moderate corruption (spelling, marriage)
    --    FNAME  0.20  — moderate specificity, higher corruption (nicknames)
    --    ZIP    0.10  — corroborating geographic signal, moderate staleness rate
    --    STATE  0.05  — corroborating geographic signal, moderate staleness rate
    --    CITY   0.10  — corroborating geographic signal, moderate staleness rate
    --    GENDER 0.05  — low discriminatory power, high recording inconsistency

    ROUND(
          SCORE_DOB    * 0.25
        + SCORE_LNAME  * 0.30
        + SCORE_FNAME  * 0.20
        + SCORE_ZIP    * 0.10
        + SCORE_STATE  * 0.05
        + SCORE_CITY   * 0.05
        + SCORE_GENDER * 0.05
    , 4) AS COMPOSITE_SCORE,

    -- Final score after consideration of deterministic override (intially this is redundant)
    CASE
        WHEN 
              SCORE_DOB    = 1
          AND SCORE_LNAME  = 1
          AND SCORE_FNAME  = 1
          AND SCORE_ZIP    = 1
          AND SCORE_STATE  = 1
          AND SCORE_CITY   = 1
          AND SCORE_GENDER = 1
        THEN 1.0   -- deterministic win: ignore probabilistic score entirely
        ELSE ROUND(
               SCORE_DOB    * 0.25
        + SCORE_LNAME  * 0.30
        + SCORE_FNAME  * 0.20
        + SCORE_ZIP    * 0.10
        + SCORE_STATE  * 0.05
        + SCORE_CITY   * 0.05
        + SCORE_GENDER * 0.05
    , 4)
    END AS FINAL_SCORE,

    -- Three-tier match decision 
    CASE
        WHEN 
              SCORE_DOB    = 1
          AND SCORE_LNAME  = 1
          AND SCORE_FNAME  = 1
          AND SCORE_ZIP    = 1
          AND SCORE_STATE  = 1
          AND SCORE_CITY   = 1
          AND SCORE_GENDER = 1                 THEN 'AUTO_MATCH'    -- deterministic
        WHEN ROUND(
               SCORE_DOB    * 0.25
        + SCORE_LNAME  * 0.30
        + SCORE_FNAME  * 0.20
        + SCORE_ZIP    * 0.10
        + SCORE_STATE  * 0.05
        + SCORE_CITY   * 0.05
        + SCORE_GENDER * 0.05
    , 4) >= 0.85                               THEN 'AUTO_MATCH'    -- high confidence
        WHEN ROUND(
                SCORE_DOB    * 0.25
        + SCORE_LNAME  * 0.30
        + SCORE_FNAME  * 0.20
        + SCORE_ZIP    * 0.10
        + SCORE_STATE  * 0.05
        + SCORE_CITY   * 0.05
        + SCORE_GENDER * 0.05
    , 4) >= 0.65                               THEN 'REVIEW'        -- steward queue
        ELSE                                        'AUTO_NO_MATCH' -- different patients
    END AS MATCH_DECISION

FROM FIELD_SCORES
-- Only retain pairs at or above the review threshold, or deterministic matches.
-- Pairs below 0.65 with no deterministic signal are discarded here —
-- they will never appear in the review queue or contribute to a cluster.
WHERE (
    (SCORE_DOB    = 1
          AND SCORE_LNAME  = 1
          AND SCORE_FNAME  = 1
          AND SCORE_ZIP    = 1
          AND SCORE_STATE  = 1
          AND SCORE_CITY   = 1
          AND SCORE_GENDER = 1)  
    OR ROUND(
                SCORE_DOB    * 0.25
        + SCORE_LNAME  * 0.30
        + SCORE_FNAME  * 0.20
        + SCORE_ZIP    * 0.10
        + SCORE_STATE  * 0.05
        + SCORE_CITY   * 0.05
        + SCORE_GENDER * 0.05
    , 4) >= 0.65
);

## #6  DATA QUALITY RULES
***Input:***  Standardized records from §1
***Output:*** One row per issue detected, with severity and field context

***Pattern:*** 
LATERAL FLATTEN over an ARRAY_CONSTRUCT of CASE expressions.

Each CASE either returns an issue object or NULL.
FLATTEN unpacks the array, and the WHERE clause drops NULLs.
This produces one output row per issue per record — multiple issues on one record produce multiple rows, all linkable by STD_RECORD_ID.

     Severity levels:
       CRITICAL — record cannot be matched reliably; route to steward
       HIGH     — significant field gap; degrades match quality
       MEDIUM   — corroborating field missing; minor match quality impact
       LOW      — cosmetic or low-signal field issue

In [ ]:
SELECT
    DIM_PATIENT_KEY,
    ---SOURCE_SYSTEM_ID,
    issue.value:ISSUE_CODE::VARCHAR        AS ISSUE_CODE,
    issue.value:ISSUE_SEVERITY::VARCHAR    AS ISSUE_SEVERITY,
    issue.value:FIELD_NAME::VARCHAR        AS FIELD_NAME,
    issue.value:FIELD_VALUE::VARCHAR       AS FIELD_VALUE,
    issue.value:ISSUE_DESCRIPTION::VARCHAR AS ISSUE_DESCRIPTION

FROM STAN_NORM,

LATERAL FLATTEN(INPUT => ARRAY_CONSTRUCT(

    -- Missing last name → HIGH
    CASE WHEN ln_std IS NULL
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'MISSING_LAST_NAME',
            'ISSUE_SEVERITY',    'HIGH',
            'FIELD_NAME',        'LAST_NAME_STD',
            'FIELD_VALUE',       NULL,
            'ISSUE_DESCRIPTION', 'Last name is missing — record will not enter primary blocking pass')
        ELSE NULL END,

    -- Missing or unparseable DOB → CRITICAL
    CASE WHEN PATIENT_DOB IS NULL
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'MISSING_DOB',
            'ISSUE_SEVERITY',    'CRITICAL',
            'FIELD_NAME',        'DATE_OF_BIRTH',
            'FIELD_VALUE',       NULL,
            'ISSUE_DESCRIPTION', 'DOB is missing or could not be parsed — highest-weight match field is zero')
        ELSE NULL END,

    -- DOB in the future → CRITICAL (data entry error or test record)
    CASE WHEN PATIENT_DOB IS NOT NULL
          AND PATIENT_DOB > CURRENT_DATE()
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'FUTURE_DOB',
            'ISSUE_SEVERITY',    'CRITICAL',
            'FIELD_NAME',        'DATE_OF_BIRTH',
            'FIELD_VALUE',       PATIENT_DOB::VARCHAR,
            'ISSUE_DESCRIPTION', 'DOB is in the future — likely data entry error or test record')
        ELSE NULL END,

    -- Implausible age (> 130 years) → HIGH
    -- Common cause: default date 1900-01-01 used when DOB unknown
    CASE WHEN PATIENT_DOB IS NOT NULL
          AND DATEDIFF('year', PATIENT_DOB, CURRENT_DATE()) > 130
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'IMPLAUSIBLE_AGE',
            'ISSUE_SEVERITY',    'HIGH',
            'FIELD_NAME',        'DATE_OF_BIRTH',
            'FIELD_VALUE',       PATIENT_DOB::VARCHAR,
            'ISSUE_DESCRIPTION', 'Age > 130 years — likely placeholder date (e.g. 1900-01-01)')
        ELSE NULL END,

    -- Invalid or missing gender code → MEDIUM
    CASE WHEN PATIENT_GENDER IS NULL OR PATIENT_GENDER NOT IN ('Male','Female','Unknown','Other')
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'INVALID_GENDER',
            'ISSUE_SEVERITY',    'MEDIUM',
            'FIELD_NAME',        'GENDER_STD',
            'FIELD_VALUE',       PATIENT_GENDER,
            'ISSUE_DESCRIPTION', 'Gender code is missing or outside expected values Male/Female/Unknown/Other')
        ELSE NULL END,

    -- ZIP not exactly 5 digits → MEDIUM
    CASE WHEN PATIENT_ZIPCODE IS NOT NULL
          AND NOT REGEXP_LIKE(PATIENT_ZIPCODE, '^[0-9]{5}$')
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'INVALID_ZIP',
            'ISSUE_SEVERITY',    'MEDIUM',
            'FIELD_NAME',        'ZIP_CODE',
            'FIELD_VALUE',       PATIENT_ZIPCODE,
            'ISSUE_DESCRIPTION', 'ZIP code is not a valid 5-digit value after standardization')
        ELSE NULL END
-- STATE
-- CITY
    
)) AS issue

WHERE issue.value IS NOT NULL;   -- drop the NULL entries (fields that passed their check)

## #9  SINGLETON DETECTION AND ROUTING

Records that land in no block under any blocking key in #2 cannot be matched. They are not errors — they are a routing decision.

Route by reason:
NULL_KEY_FIELD   → DQ queue (fix the data, re-ingest)
RARE_DEMOGRAPHICS, high DQ → auto-provision as new master patient
RARE_DEMOGRAPHICS, low DQ  → steward review

Singletons are re-evaluated on every pipeline run — a singleton today may find a match when a second record from the same source arrives tomorrow. Do not permanently discard them.

***IDENTIFICATION APPROACH*** — aggregation, not joining:

A record is a singleton if it matches no other record on ANY blocking key. Instead of joining to find matches, aggregate each
blocking key independently to count how many records share each key value. A record is a singleton if its key value count = 1
 (unique value) OR its key is NULL (no value at all).

Each CTE below is a single GROUP BY scan — O(n) not O(n²).
The final join is each record against three small lookup tables, not against the full record table.

***Complexity:*** O(n) for each key aggregation → O(3n) total.

***Previous:***   O(n²) for the self-join.

On 1M records: ~3M row operations vs ~1 trillion. Order of magnitude improvement in practice even accounting for Snowflake parallelism.


In [ ]:
WITH
 -- Step 1: Count records per blocking key value 
--   One GROUP BY per key. These are cheap full-table aggregations.
--   Result: for each distinct key value, how many records share it?
--   Key value with count = 1 means that value is unique → any record
--   holding it has no block-mates under this key.

PRIMARY_BLOCK AS (
-- LNSDX_ZIP
    SELECT
        ln_sdx || '|' || PATIENT_ZIPCODE::VARCHAR  AS KEY_VALUE,
        COUNT(*)                               AS KEY_COUNT
    FROM STAN_NORM
    WHERE 
          ln_sdx IS NOT NULL
      AND PATIENT_ZIPCODE IS NOT NULL
    GROUP BY 
        ln_sdx,
        PATIENT_ZIPCODE
),
 
FALLBACK_BLOCK AS (
-- LNSDX_DOB
    SELECT
        ln_sdx || '|' || PATIENT_DOB::VARCHAR   AS KEY_VALUE,
        COUNT(*)                                AS KEY_COUNT
    FROM STAN_NORM
    WHERE ln_sdx           IS NOT NULL
      AND PATIENT_DOB      IS NOT NULL
    GROUP BY ln_sdx, PATIENT_DOB
),
 
LAST_RESORT_BLOCK AS (
-- ZIP3_DOB
    SELECT
        PATIENT_DOB::VARCHAR || '|' || ZIP3        AS KEY_VALUE,
        COUNT(*)                                   AS KEY_COUNT
    FROM STAN_NORM
    WHERE PATIENT_DOB IS NOT NULL
      AND ZIP3        IS NOT NULL
    GROUP BY PATIENT_DOB, ZIP3
),
 
-- Step 2: For each record, look up whether any key has count > 1 
--   LEFT JOIN each record against the three count tables.
--   If the count for a key is NULL (no match = key was NULL on this record)
--   or = 1 (unique value = no other record shares this key), that key
--   cannot place the record in a block with anyone else.
--   A record is a singleton only if ALL THREE keys are NULL or unique.
 
KEYED AS (
    SELECT
        s.DIM_PATIENT_KEY,
        --s.SOURCE_SYSTEM_ID,
        s.ln_std,
        s.fn_std,
        s.PATIENT_DOB,
        s.PATIENT_ZIPCODE,
        --s.DQ_SCORE_OVERALL,
        s.ln_sdx,
        s.fn_sdx,
 
        -- LNSDX_ZIP
        pri.KEY_COUNT    AS LNSDX_ZIP_PEERS,
 
        -- LNSDX_DOB
        fb.KEY_COUNT    AS LNSDX_DOB_PEERS,
 
        -- ZIP3_DOB
        lr.KEY_COUNT    AS ZIP3_DOB_PEERS
 
    FROM STAN_NORM AS s
 
    LEFT JOIN PRIMARY_BLOCK AS pri
        ON pri.KEY_VALUE = s.ln_sdx || '|' || s.PATIENT_ZIPCODE::VARCHAR 
        AND s.ln_sdx           IS NOT NULL
        AND s.PATIENT_ZIPCODE  IS NOT NULL

    
    LEFT JOIN FALLBACK_BLOCK AS fb
        ON fb.KEY_VALUE = s.ln_sdx || '|' || s.PATIENT_DOB::VARCHAR
        AND s.ln_sdx IS NOT NULL
        AND s.PATIENT_DOB     IS NOT NULL
 
    LEFT JOIN LAST_RESORT_BLOCK AS lr
        ON lr.KEY_VALUE = s.PATIENT_DOB::VARCHAR || '|' || s.ZIP3
        AND s.PATIENT_DOB IS NOT NULL
        AND s.ZIP3        IS NOT NULL
)
 
SELECT
    DIM_PATIENT_KEY,
    --SOURCE_SYSTEM_ID,
    ln_std,
    fn_std,
    PATIENT_DOB,
    PATIENT_ZIPCODE,
    --DQ_SCORE_OVERALL,
 
    -- Expose peer counts for transparency — useful for steward review
    COALESCE(LNSDX_ZIP_PEERS, 0)        AS LNSDX_ZIP_PEERS,
    COALESCE(LNSDX_DOB_PEERS, 0)        AS LNSDX_DOB_PEERS,
    COALESCE(ZIP3_DOB_PEERS,  0)        AS ZIP3_DOB_PEERS,
 
    -- Classify the reason for being a singleton.
    -- Ordered from most actionable (fixable data problem) to least
    -- (genuinely rare demographics — not necessarily a data problem).
    CASE
        WHEN ln_std  IS NULL
         AND fn_std IS NULL                            THEN 'NULL_BOTH_NAMES'
        WHEN ln_std  IS NULL                           THEN 'NULL_LAST_NAME'
        WHEN fn_std IS NULL                            THEN 'NULL_FIRST_NAME'
        WHEN PATIENT_DOB  IS NULL                      THEN 'NULL_DOB'
        WHEN PATIENT_ZIPCODE IS NULL                   THEN 'NULL_ZIP'
        ELSE                                                'RARE_DEMOGRAPHICS'
    END AS SINGLETON_REASON,
 
    -- Routing decision
    CASE
        WHEN ln_std           IS NULL
          OR PATIENT_ZIPCODE  IS NULL
          OR PATIENT_DOB      IS NULL                            THEN 'DQ_QUEUE'
        --WHEN DQ_SCORE_OVERALL >= 70                            THEN 'AUTO_PROVISION_MPI'
        ELSE                                                          'STEWARD_REVIEW'
    END AS SUGGESTED_ROUTE,
 
    CURRENT_TIMESTAMP() AS DETECTED_AT
 
FROM KEYED
WHERE
    -- A record is a singleton if it has no block-mates under ANY key.
    -- No block-mates = key is NULL (LEFT JOIN returned nothing)
    --               OR key count = 1 (only this record holds that value)
    COALESCE(LNSDX_ZIP_PEERS, 0) <= 1
    AND COALESCE(LNSDX_DOB_PEERS,  0) <= 1
    AND COALESCE(ZIP3_DOB_PEERS,    0) <= 1;
 

# #10 GROUND TRUTH BOOTSTRAP
How to assemble known match pairs without SSN, to measure PC in #7 and calibrate thresholds in #4.

True Match on First Name, Last Name DOB, Gender, City, State and Zip are assumed 'perfect matches'

In [ ]:
%%sql -r dataframe_1
SELECT DISTINCT
    A.DIM_PATIENT_KEY AS A_KEY,
    B.DIM_PATIENT_KEY AS B_KEY,
    A.MRN AS A_MRN,B.MRN AS B_MRN,
    A.IDENTIFIER AS A_IDN,B.IDENTIFIER AS B_IDN,
    A.FIRST_NAME AS A_FN,B.FIRST_NAME AS B_FN,
    A.LAST_NAME AS A_LN,B.LAST_NAME AS B_LN,
    A.PATIENT_DOB AS A_DOB,B.PATIENT_DOB AS B_DOB,
    A.PATIENT_GENDER AS A_GEND,B.PATIENT_GENDER AS B_GEND,
    A.PATIENT_ADDRESS1 AS A_ADD, B.PATIENT_ADDRESS1 AS B_ADD,
    A.PATIENT_CITY AS A_CIT,B.PATIENT_CITY AS B_CIT,
    A.PATIENT_STATE AS A_ST,B.PATIENT_STATE AS B_ST,
    A.PATIENT_ZIPCODE AS A_ZIP,B.PATIENT_ZIPCODE AS B_ZIP,
  FROM STAN_NORM AS A
  JOIN STAN_NORM AS B
    ON A.DIM_PATIENT_KEY < B.DIM_PATIENT_KEY
    AND A.FIRST_NAME=B.FIRST_NAME 
    AND A.LAST_NAME=B.LAST_NAME
    AND A.PATIENT_DOB=B.PATIENT_DOB
    AND A.PATIENT_GENDER=B.PATIENT_GENDER
    AND A.PATIENT_CITY=B.PATIENT_CITY
    AND A.PATIENT_STATE=B.PATIENT_STATE
    AND A.PATIENT_ZIPCODE=B.PATIENT_ZIPCODE
ORDER BY A_KEY DESC;